# 11 · 地理空间与图论

用 Python 把「位置」与「关系」化为可计算的数据：先学几何与地理座标的表达与运算，再学图论 graph 对网络结构的建模与分析。

## 学习目标
- 能用 shapely 创建并操作矢量几何 vector geometry（点、多边形、外置矩形），计算面积与空间关系。
- 理解座标参考系统 CRS（Coordinate Reference System）概念，知道面积与距离何时需要座标转换。
- 能把表格数据组成 GeoDataFrame 概念模型并依属性上色绘图。
- 认识 OpenStreetMap（OSM）开放地图数据与其标签 tag 系统，理解 osmnx 取得建筑与路网的流程，以及脱机备援与数据授权。
- 能用空间索引 spatial index 与最近邻 nearest neighbor 查找加速大量点的搜索。
- 能用 networkx 创建图、计算中心性 centrality 并侦测社区 community。

In [ ]:
# 概念：环境与绘图字体 setup（本书会画地图与图论结构，先统一设置）
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 注意：matplotlib 预设字体无中文字，设一个常见中文字体避免图上中文变方框
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 让负号正常显示，不被字体吃掉

np.random.seed(11)   # 固定乱数种子，让每次重跑的仿真数据一致，方便对照
print("setup 完成，numpy 版本：", np.__version__)

## 矢量几何基础 shapely

矢量几何 vector geometry 用座标精确描述空间对象：点 Point、线、多边形 Polygon。它是地理与空间问题的最小积木。

为什么先学它：后续所有空间运算（面积、相交、距离）都需要先有「几何对象」当对象。先掌握如何用座标构造几何，运算才有施力点。

常见几何与属性：

| 对象 | 创建方式 | 常用属性 |
|---|---|---|
| 点 Point | `Point(x, y)` | `.x`、`.y` |
| 多边形 Polygon | `Polygon([(x, y), ...])` | `.area`、`.exterior`、`.bounds` |
| 外置矩形 box | `box(minx, miny, maxx, maxy)` | 同 Polygon |

In [ ]:
# 概念：用座标手动创建 Point 与 Polygon，查看其座标与基本属性
from shapely.geometry import Point, Polygon, box

# 自造几个点（座标可想成公尺平面上的位置）
p1 = Point(0, 0)
p2 = Point(3, 4)
print("p2 座标：", p2.x, p2.y)
print("p1 到 p2 的距离：", p1.distance(p2))   # 直线距离，毕氏定理结果应为 5

# 概念：多边形用「外环座标串行」定义，首尾会自动闭合
square = Polygon([(0, 0), (4, 0), (4, 4), (0, 4)])   # 一个 4x4 的正方形
print("square 面积 area：", square.area)
print("square 外置范围 bounds：", square.bounds)       # (minx, miny, maxx, maxy)

# 边界 boundary 是外环线，外环 exterior 是构成外圈的线串
print("外环 exterior 座标：", list(square.exterior.coords))

# 技巧：box 是创建轴对齐矩形的捷径，等价于手写四个角
rect = box(1, 1, 3, 2)                               # minx,miny,maxx,maxy
print("box 面积：", rect.area)

## 几何运算与空间索引

有了几何对象，就能量测（面积 area、长度）与判断关系：空间关系 spatial predicate 包含 contains、相交 intersects、相离等。仿射变换 affinity 则可平移、缩放、旋转几何。

为什么需要空间索引 spatial index：当几何数量很多时，逐一两两比对是 O(n²)。STRtree 先用外置框做「粗筛 coarse filter」找出可能相交的候选，再精算，避免无谓比对。

形状（不运行，仅示意）：

```
tree = STRtree(几何清单)
候选索引 = tree.query(查找几何)   # 只回传外置框可能相交者
```

In [ ]:
# 概念：空间关系判断与仿射变换
from shapely.geometry import Point, Polygon, box
from shapely import affinity

plot = box(0, 0, 10, 10)                  # 一块基地
house = box(2, 2, 5, 6)                   # 基地上的一栋建筑

print("基地是否包含建筑 contains：", plot.contains(house))
print("两者是否相交 intersects：", plot.intersects(house))
print("基地是否包含某点：", plot.contains(Point(1, 1)))

# 仿射变换：把建筑往右上平移，再放大 1.5 倍（仿真量体调整）
moved = affinity.translate(house, xoff=3, yoff=1)        # 平移
scaled = affinity.scale(house, xfact=1.5, yfact=1.5)     # 以中心缩放
print("平移后 bounds：", moved.bounds)
print("放大后面积：", scaled.area, "（原面积", house.area, "）")

In [ ]:
# 概念：用 STRtree 对数十个小多边形做粗筛，并与暴力法比对结果
from shapely.geometry import box
from shapely.strtree import STRtree

# 自造 30 个随机小方块当作建筑足迹
boxes = []
for _ in range(30):
    x, y = np.random.uniform(0, 50, size=2)              # 随机左下角
    boxes.append(box(x, y, x + 3, y + 3))                # 边长 3 的小方块

query_window = box(10, 10, 25, 25)                       # 一个查找框

tree = STRtree(boxes)                                    # 建空间索引
# 注意：新版 shapely 的 query 回传「索引」数组，不是几何本身
candidate_idx = tree.query(query_window)
# 粗筛只保证外置框可能相交，仍需精算真正相交者
index_hits = [i for i in candidate_idx if boxes[i].intersects(query_window)]

# 暴力法：逐一比对全部 30 个，作为正确答案对照
brute_hits = [i for i, b in enumerate(boxes) if b.intersects(query_window)]

print("STRtree 粗筛候选数：", len(candidate_idx))
print("索引精算相交数：", len(index_hits))
print("暴力法相交数：", len(brute_hits))
print("两法结果一致：", sorted(index_hits) == sorted(brute_hits))

## 地理座标系统与座标转换

座标参考系统 CRS（Coordinate Reference System）定义一组座标数字对应地球上哪个位置。同一个点在不同 CRS 下数值完全不同。

两大类常见 CRS：

| 类型 | 例子 | 单位 | 适合做什么 |
|---|---|---|---|
| 地理座标 geographic | EPSG:4326（经纬度） | 度 degree | 定位、访问、交换数据 |
| 投影座标 projected | EPSG:3857 等（公尺） | 公尺 metre | 量面积、量距离 |

为什么重要：面积与距离只有在投影座标下才有正确的公尺意义。直接拿经纬度（度数）算面积是错的。「何时该转换」是地理分析的关键观念。

In [ ]:
# 概念：同一点在地理座标与投影座标下的数值差异（pyproj 转换）
from pyproj import Transformer

# 自造两个经纬度点（lon 经度, lat 纬度），约在台北一带
points_lonlat = [(121.5654, 25.0330), (121.5170, 25.0478)]

# 注意：always_xy=True 让输入输出都用 (x=经度, y=纬度) 顺序，避免常见的轴颠倒坑
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

for lon, lat in points_lonlat:
    x, y = transformer.transform(lon, lat)               # 度 -> 公尺
    print(f"经纬度 ({lon}, {lat}) -> 投影公尺 ({x:.1f}, {y:.1f})")

# 技巧：两点距离在投影座标下才有公尺意义；用转换后座标算才正确
(x0, y0), (x1, y1) = [transformer.transform(lon, lat) for lon, lat in points_lonlat]
dist_m = ((x1 - x0) ** 2 + (y1 - y0) ** 2) ** 0.5
print("两点投影距离（公尺，含投影变形）：", round(dist_m, 1))

## GeoDataFrame 与地图绘制 geopandas

GeoDataFrame 是 geopandas 的核心：一张表格 DataFrame 多了一个特殊的 geometry 栏，存放每列的几何对象，并记录整体的 CRS。

为什么好用：它把熟悉的表格操作（筛选、聚合）扩充到几何，让属性数据与位置数据一起处理与可视化。依某个属性栏上色画图，就是主题地图 choropleth 的概念。

形状（不运行，仅示意）：

```
gdf = GeoDataFrame(属性表, geometry=几何清单, crs="EPSG:3857")
gdf.plot(column="属性栏", legend=True)   # 依属性上色
```

In [ ]:
# 概念：自造区域多边形组成 GeoDataFrame，依数值上色画主题地图
import geopandas as gpd
from shapely.geometry import box

# 自造四个相邻区域（仿真都市分区）与一栏人口密度数值
regions = [box(0, 0, 2, 2), box(2, 0, 4, 2), box(0, 2, 2, 4), box(2, 2, 4, 4)]
names = ["A", "B", "C", "D"]
density = [120, 340, 80, 500]                            # 仿真人口密度（人/公顷）

# 用几何清单与属性建 GeoDataFrame，并标记 CRS 为投影座标（公尺）
gdf = gpd.GeoDataFrame(
    {"name": names, "density": density},
    geometry=regions,
    crs="EPSG:3857",
)
print(gdf)
print("目前 CRS：", gdf.crs)
print("各区面积（由几何自动算）：", list(gdf.area))

# 依 density 栏上色：数值高的区域颜色较深，即 choropleth
ax = gdf.plot(column="density", cmap="OrRd", legend=True, edgecolor="black")
ax.set_title("各分区人口密度主题地图 choropleth")
plt.show()

## OpenStreetMap 真实地理数据 osmnx

OpenStreetMap（OSM）是开放的全球地图数据库，用标签 tag（key=value 形式）描述世界，例如 `building=yes`、`highway=primary`、`building:levels=5`、`height=18`。osmnx 能依地名 place 或范围下载建筑足迹 footprint 与路网 street network，回传成 GeoDataFrame。

工程意识（很重要）：
- 下载需连网，且 OSM 数据受授权 license 规范，使用需标注来源。
- 网络不稳时应有脱机 CSV 备援 offline fallback，流程才稳定。
- 标签常缺漏（并非每栋都有 `building:levels`），要先想好默认值与缺值处理。

下载形状（不运行，仅示意）：

```
gdf = osmnx.features_from_place("地名", tags={"building": True})
G = osmnx.graph_from_place("地名", network_type="drive")
```

In [ ]:
# 概念：用内置仿真标签表示意 OSM 流程（不强制连网，含缺值与用途分类）
import pandas as pd

# 注意：真实情境会用 osmnx 下载；这里用一张内置仿真标签表当脱机备援，
#       让流程在无网络时仍可示范。每列代表一栋建筑的 OSM 标签。
osm_like = pd.DataFrame([
    {"id": 1, "building": "residential", "building:levels": "5",  "height": None},
    {"id": 2, "building": "commercial",  "building:levels": None, "height": "24"},
    {"id": 3, "building": "yes",         "building:levels": "3",  "height": None},
    {"id": 4, "building": "retail",      "building:levels": None, "height": None},
])

# 一张小型标签对照表：把 building 标签映射成用途分类 classification
use_map = {"residential": "住宅", "commercial": "商业", "retail": "商业", "yes": "未知"}
osm_like["use"] = osm_like["building"].map(use_map).fillna("未知")

# 缺值处理：building:levels 缺时，若有 height 就用 height/3 估楼层，否则预设 1 层
def estimate_levels(row):
    if pd.notna(row["building:levels"]):
        return int(row["building:levels"])               # 标签有值，直接用
    if pd.notna(row["height"]):
        return max(1, round(float(row["height"]) / 3))   # 用高度粗估，每层约 3 公尺
    return 1                                              # 两者皆缺，保守给 1 层

osm_like["levels_est"] = osm_like.apply(estimate_levels, axis=1)
print(osm_like[["id", "building", "use", "levels_est"]])

## 最近邻查找 cKDTree

最近邻 nearest neighbor 查找：在一堆点中找「离某点最近的几个点」。scipy 的 cKDTree 把点建成 KD 树 KD-tree，让重复查找的成本远低于每次都暴力扫全部。

常见查找：
- k 最近邻 k-nearest neighbor：`tree.query(点, k=k)` 回传最近 k 个的距离与索引。
- 半径范围查找：`tree.query_ball_point(点, r)` 回传距离 r 内所有点的索引。

为什么学：空间配对、聚合、群聚分析都创建在「快速找邻居」之上。

In [ ]:
# 概念：用 cKDTree 查每个点的最近邻，并与暴力法验证
from scipy.spatial import cKDTree

# 自造一批二维点（仿真建筑点位）
pts = np.random.uniform(0, 100, size=(20, 2))

tree = cKDTree(pts)                                       # 一次建树，可重复查找
# 查每个点最近的 2 个（k=2，因为第 1 个一定是它自己，距离 0）
dists, idx = tree.query(pts, k=2)
nearest_idx = idx[:, 1]                                   # 第 2 栏才是真正的最近邻
nearest_dist = dists[:, 1]

# 暴力法验证第 0 号点的最近邻
diff = pts - pts[0]
all_d = np.sqrt((diff ** 2).sum(axis=1))
all_d[0] = np.inf                                         # 排除自己
brute_nearest = int(all_d.argmin())

print("第 0 号点 KD 树最近邻索引：", nearest_idx[0])
print("第 0 号点暴力法最近邻索引：", brute_nearest)
print("一致：", nearest_idx[0] == brute_nearest)

# 半径查找：找出第 0 号点 30 单位内的所有邻居
within = tree.query_ball_point(pts[0], r=30)
print("第 0 号点 30 单位内的点数（含自己）：", len(within))

## 图论建模 networkx

图 graph 由节点 node 与边 edge 组成，用来描述「实体之间的关系」。交通网、社交关系、依赖关系本质都是图。

几个关键区分：
- 无向图 undirected：边没有方向（朋友关系）；有向图 directed：边有方向（追踪关系）。
- 权重 weight：边可附数值（距离、成本）。
- 邻接 adjacency：某节点直接相连的邻居集合；度 degree 是邻居数量。

形状（不运行，仅示意）：

```
G = networkx.Graph()
G.add_edge(a, b, weight=w)   # 从边清单逐条加入
```

In [ ]:
# 概念：用边清单创建小型无向加权图，查看节点、边与度
import networkx as nx

# 自造一份边清单：(节点A, 节点B, 权重) 仿真建筑间的邻近距离
edges = [
    ("A", "B", 5), ("A", "C", 8), ("B", "C", 3),
    ("C", "D", 6), ("D", "E", 2), ("C", "E", 9),
]

G = nx.Graph()                                           # 无向图
for a, b, w in edges:
    G.add_edge(a, b, weight=w)                           # 加边时节点自动创建

print("节点 nodes：", list(G.nodes))
print("边 edges：", list(G.edges(data="weight")))
print("各节点的度 degree：", dict(G.degree()))            # 每个节点的邻居数

# 画出图结构，边上标权重
pos = nx.spring_layout(G, seed=11)                       # 固定布局，重跑位置一致
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=800)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "weight"))
plt.title("小型无向加权图")
plt.show()

## 图的分析：中心性与社区

建好图后，常要回答「谁是关键节点」与「图如何自然聚类」。中心性 centrality 把节点的重要性量化成数字：

| 中心性 | 衡量什么 |
|---|---|
| 度中心性 degree | 直接连到多少节点（人脉广度） |
| 介数中心性 betweenness | 多少最短路径 shortest path 经过它（桥梁角色） |
| 接近中心性 closeness | 到其他所有节点的平均距离有多近 |

社区侦测 community detection 则找出「内部链接密、彼此链接疏」的节点群，揭示图的自然分组。

In [ ]:
# 概念：在前一单元的图上算三种中心性并做一次社区划分
import networkx as nx

# 重建同一张图（让本 cell 自足，不依赖上一格变量）
edges = [("A", "B", 5), ("A", "C", 8), ("B", "C", 3),
         ("C", "D", 6), ("D", "E", 2), ("C", "E", 9)]
G = nx.Graph()
for a, b, w in edges:
    G.add_edge(a, b, weight=w)

deg_c = nx.degree_centrality(G)
bet_c = nx.betweenness_centrality(G)
clo_c = nx.closeness_centrality(G)

print("度中心性 degree：   ", {k: round(v, 2) for k, v in deg_c.items()})
print("介数中心性 betweenness：", {k: round(v, 2) for k, v in bet_c.items()})
print("接近中心性 closeness： ", {k: round(v, 2) for k, v in clo_c.items()})

# 找出最关键节点：介数最高者通常是连接不同区块的桥梁
key_node = max(bet_c, key=bet_c.get)
print("介数最高（最关键）节点：", key_node)

# 社区侦测：greedy modularity 把节点分成内部紧密的群
communities = nx.community.greedy_modularity_communities(G)
print("社区划分结果：", [sorted(c) for c in communities])

## 延伸工具概观 raster 栅格

前面都在用矢量数据 vector（用座标精确描述边界）。另一大类是栅格数据 raster：把空间切成规则网格，每个像元 pixel 存一个值，像照片或热度图。

两者长处对照：

| 面向 | 矢量 vector | 栅格 raster |
|---|---|---|
| 表达 | 精确边界、面积 | 连续场、密度、覆盖率 |
| 适合 | 行政界、建筑轮廓 | 高程、温度、密度图、叠图分析 |
| 代表工具 | shapely / geopandas | rasterio |

栅格化 rasterize 就是把矢量几何「烧进」网格（rasterio 可做）。何时用：要做密度图或多层叠图分析时，网格比逐几何比对更直接。本单元只创建概念，用简单网格示意。

In [ ]:
# 概念：用 numpy 网格示意「把建筑多边形转成覆盖率网格」（rasterize 概念）
from shapely.geometry import box

# 自造两栋建筑足迹，放在 0~10 的平面上
buildings = [box(1, 1, 4, 3), box(6, 5, 9, 9)]

# 建一个 10x10 的网格，每格 1x1；逐格判断中心点是否落在任一建筑内
grid = np.zeros((10, 10), dtype=int)
for row in range(10):
    for col in range(10):
        center = Point(col + 0.5, row + 0.5)             # 该格中心
        # 注意：这是示意用的逐格扫描；真实大数据应用 rasterio 高效栅格化
        if any(b.contains(center) for b in buildings):
            grid[row, col] = 1                           # 1 表示此格被建筑覆盖

print("被建筑覆盖的格数：", grid.sum(), "/ 100")
plt.imshow(grid, origin="lower", cmap="Greys")           # origin=lower 让 y 向上
plt.title("建筑覆盖率栅格 raster 示意")
plt.colorbar(label="是否被覆盖")
plt.show()

## 练习

以下三题由浅入深，皆为复合型但简短。数据自己用 numpy / list 造，不引用任何外部文件。

In [ ]:
# TODO 1 ·（简单）基地占地面积与建蔽率（集成：多边形 Polygon + 面积 area）
#   情境：手上有一块街廓基地与其上一栋建筑的轮廓，想算出建蔽率
#         coverage（建筑占地 footprint 除以基地面积）。
#   要求：
#     1. 用 shapely 的 Polygon，以自己写死的几组座标（list）各建一个基地多边形
#        与一个建筑轮廓多边形。
#     2. 分别取两者的 .area，相除得到建蔽率。
#   验收：应该看到 print 出基地面积、建筑占地面积，以及一个介于 0 与 1 之间的
#         建蔽率数值。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）街廓网格的楼地板面积聚合
#   （集成：点 Point + 包含 contains + 循环 / cKDTree 聚合 + 属性汇总）
#   情境：城市里有一批建筑点位，各自带有楼层数 floors 与占地面积 footprint，
#         想把它们聚合到所属的街廓网格 cell，估算每格的楼地板面积
#         GFA（Gross Floor Area，= 占地面积 x 楼层数）。
#   要求：
#     1. 用 numpy 自造数十栋建筑的 (x, y) 座标、floors、footprint 三个数组
#        （仿真数字），并用一组座标切出 2x2 的方形网格 cell
#        （可用 shapely box 或座标范围判断）。
#     2. 对每栋建筑算出 GFA，并判断它落在哪一个 cell（用 box.contains(Point)
#        或座标比较）。
#     3. 把同一 cell 内所有建筑的 GFA 加总，得到每格的总楼地板面积。
#   验收：应该看到 print 出 4 个网格各自的总 GFA，且所有网格 GFA 相加等于
#         全部建筑 GFA 总和。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）高度政策情境的邻近影响与关键节点
#   （集成：networkx 建图 + 中心性 centrality + 条件聚合 + 情境前后比较）
#   情境：把彼此距离够近的建筑视为「邻近关系」连成一张图 graph，评估一项放宽
#         高度的政策（对特定用途分类的建筑套用楼高 height 乘数）后，哪些建筑
#         因为位置与邻里链接最关键、且受政策影响最大。
#   要求：
#     1. 用 numpy 自造一批建筑的 (x, y) 座标、楼高 height，以及用途分类标签
#        use（例如 0=住宅、1=商业，仿真即可）。
#     2. 以「两栋建筑距离小于门槛 d 即连一条边」建出 networkx 无向图，并计算
#        每个节点的度中心性 degree centrality 与介数中心性 betweenness centrality。
#     3. 套用政策情境：对 use==1（商业）的建筑把 height 乘以一个放宽乘数，算出
#        每栋的高度变化量，再以「中心性 x 高度变化量」作为综合影响分数排序。
#   验收：应该看到 print 出政策前后的高度对照，以及综合影响分数最高的前 3 栋
#         建筑（同时兼具高中心性与显著高度增幅）。
# TODO: 在这里完成

## 小结

- 矢量几何 vector geometry 是空间问题的最小积木：shapely 的 Point、Polygon、box 用座标构造几何，并提供面积 area、bounds、exterior 等属性。
- 几何运算包含空间关系 spatial predicate（contains、intersects）与仿射变换 affinity；几何很多时用 STRtree 做粗筛避免 O(n²) 暴力比对。
- 座标参考系统 CRS 决定座标数字的意义：地理座标 geographic 是度数、投影座标 projected 是公尺；面积与距离只有在投影座标下才正确，pyproj 负责转换。
- GeoDataFrame 把表格扩充出 geometry 栏与 CRS，可依属性上色画主题地图 choropleth。
- OpenStreetMap 用标签 tag 描述世界，osmnx 可取得建筑与路网；工程上要注意连网、授权 license、脱机备援与标签缺值处理。
- cKDTree 用 KD 树 KD-tree 加速最近邻 nearest neighbor 与半径查找，是空间配对与聚合的基础。
- networkx 用节点与边建模关系；中心性 centrality（度、介数、接近）量化节点重要性，社区侦测 community detection 揭示自然聚类。
- 矢量 vector 之外还有栅格 raster：用网格与像元 pixel 表达连续场，rasterio 可做栅格化 rasterize，适合密度图与叠图分析。